# 5.2 · Building a client and the LLM tool-use loop

**构建客户端与 LLM 工具调用循环**

> **After this lesson you can:** Build an MCP client that connects to a server, hands its tools to an LLM, and runs a robust multi-round agent loop with error passing, timeouts and an iteration guard.

Official reading: [Build an MCP client](https://modelcontextprotocol.io/docs/2026-07-28/develop/build-client)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. A multi-tool server to drive
Our server exposes three tools that can be chained. `find_city_code` → `get_forecast` → `convert_temperature`. `get_forecast` raises `ToolError` for unknown codes, which is a deliberate, readable failure the model can recover from.

In [ ]:
import json
import logging
import os
from types import SimpleNamespace

from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

logging.getLogger("mcp").setLevel(logging.CRITICAL)

CITIES = {"paris": "PAR", "london": "LON", "tokyo": "TYO"}
FORECASTS = {"PAR": {"today": {"celsius": 17, "rain_probability": 80}, "tomorrow": {"celsius": 21, "rain_probability": 10}},
             "LON": {"today": {"celsius": 14, "rain_probability": 95}, "tomorrow": {"celsius": 15, "rain_probability": 60}},
             "TYO": {"today": {"celsius": 26, "rain_probability": 5}, "tomorrow": {"celsius": 24, "rain_probability": 30}}}

weather = MCPServer("weather")


@weather.tool()
def find_city_code(city: str) -> str:
    """Return the 3-letter forecast code for a city name, e.g. 'Paris' -> 'PAR'."""
    try:
        return CITIES[city.strip().lower()]
    except KeyError:
        raise ToolError(f"Unknown city {city!r}. Known: {sorted(CITIES)}")


@weather.tool()
def get_forecast(code: str, day: str = "today") -> dict:
    """Forecast for a city CODE (use find_city_code first) and day ('today' or 'tomorrow')."""
    if code not in FORECASTS:
        raise ToolError(f"Unknown city code {code!r}. Call find_city_code to get a valid code.")
    return {"code": code, "day": day, **FORECASTS[code][day]}


@weather.tool()
def convert_temperature(celsius: float) -> float:
    """Convert Celsius to Fahrenheit."""
    return round(celsius * 9 / 5 + 32, 1)


print("server ready:", weather.name)

### 2. Convert MCP tools to the model's tool format
This is the one comprehension from the tutorial, plus pagination. Note what is *not* forwarded: `outputSchema` and annotations stay with the host.

In [ ]:
async def list_all_tools(client):
    tools, cursor = [], None
    while True:
        page = await client.list_tools(cursor=cursor)
        tools += page.tools
        cursor = page.next_cursor
        if cursor is None:
            return tools


def to_anthropic(tools):
    return [{"name": t.name, "description": t.description, "input_schema": t.input_schema} for t in tools]


async with Client(weather) as client:
    mcp_tools = await list_all_tools(client)
    print("MCP tool as listed   :", json.dumps(mcp_tools[1].model_dump(by_alias=True, exclude_none=True))[:400], "...\n")
    print("Anthropic tool format:", json.dumps(to_anthropic(mcp_tools)[1], indent=2))

### 3. A deterministic "fake LLM" with the same interface as the Anthropic SDK
To run offline, we need something with the Anthropic client's interface: `llm.messages.create(model=..., max_tokens=..., tools=..., messages=...)` returning an object with `stop_reason` and `content` blocks (`type`, `text`, `id`, `name`, `input`). The *policy* reads the conversation (including earlier tool results and `is_error` flags) and decides the next move. That makes the loop's behaviour visible and repeatable. Swapping in the real `anthropic.Anthropic()` later needs **no change to the loop**.

In [ ]:
def text_block(text):
    return SimpleNamespace(type="text", text=text)


_ids = iter(range(1, 10_000))
def tool_use(name, args):
    return SimpleNamespace(type="tool_use", id=f"toolu_{next(_ids):03d}", name=name, input=args)


def tool_trace(messages):
    """Pair every tool_use the 'model' made with its tool_result: [{name, input, content, is_error}, ...]."""
    calls = {}
    for m in messages:
        for block in (m["content"] if isinstance(m["content"], list) else []):
            if getattr(block, "type", None) == "tool_use":
                calls[block.id] = {"name": block.name, "input": block.input}
            elif isinstance(block, dict) and block.get("type") == "tool_result":
                calls[block["tool_use_id"]].update(content=block["content"], is_error=block["is_error"])
    return list(calls.values())


class FakeLLM:
    """Mimics anthropic.Anthropic().messages.create with a scripted policy(messages, tool_names) -> blocks."""

    def __init__(self, policy):
        self.policy, self.calls = policy, 0
        self.messages = self                              # so `llm.messages.create(...)` works

    def create(self, *, model, max_tokens, tools, messages):
        self.calls += 1
        blocks = self.policy(messages, {t["name"] for t in tools})
        stop = "tool_use" if any(b.type == "tool_use" for b in blocks) else "end_turn"
        return SimpleNamespace(stop_reason=stop, content=blocks)

Now the scripted policy for questions like *"Do I need an umbrella in Paris?"* To show **recovery**, it first *guesses* a code ("PARIS"), sees the `is_error` result, and corrects course. Tool A's output (`PAR`) then feeds tool B.

In [ ]:
import re


def umbrella_policy(messages, tool_names):
    question = messages[0]["content"] if isinstance(messages[0]["content"], str) else ""
    city = re.search(r"in (\w+)", question).group(1)
    day = "tomorrow" if "tomorrow" in question else "today"
    trace = tool_trace(messages)
    if not trace:                                                    # round 1: a careless guess
        return [text_block(f"Let me check the forecast for {city}."), tool_use("get_forecast", {"code": city.upper(), "day": day})]
    last = trace[-1]
    if last["is_error"] and last["name"] == "get_forecast":         # recover: read the error, look up the code
        return [text_block("That code was wrong; looking it up."), tool_use("find_city_code", {"city": city})]
    if last["name"] == "find_city_code" and not last["is_error"]:   # chain: A's output becomes B's input
        return [tool_use("get_forecast", {"code": last["content"], "day": day})]
    if last["name"] == "get_forecast":
        f = json.loads(last["content"])
        verdict = "Yes, take an umbrella" if f["rain_probability"] >= 50 else "No umbrella needed"
        return [text_block(f"{verdict}: {f['rain_probability']}% chance of rain in {city} {day}, {f['celsius']}°C.")]
    return [text_block(f"I could not get a forecast: {last['content']}")]


print("policy defined")

### 4. The agent loop
This is the standard manual tool-use loop for the Messages API: loop while `stop_reason == "tool_use"`, append the assistant content verbatim, then **one** user message with **all** tool results. It adds an iteration guard and turns transport errors (such as timeouts) into `is_error` results so the model can react.

In [ ]:
from mcp.shared.exceptions import MCPError
import mcp_types as types

MODEL = "claude-opus-5-5"


def text_of(result: types.CallToolResult) -> str:
    return "\n".join(b.text for b in result.content if isinstance(b, types.TextContent))


async def run_agent(client, llm, messages, *, max_iterations=6, verbose=True):
    """Run one user turn to completion. Mutates and returns `messages` (the conversation state)."""
    tools = to_anthropic(await list_all_tools(client))
    for step in range(1, max_iterations + 1):
        response = llm.messages.create(model=MODEL, max_tokens=16000, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": response.content})
        for block in response.content:
            if block.type == "text" and verbose:
                print(f"  [{step}] model: {block.text}")
        if response.stop_reason != "tool_use":
            return messages
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            try:
                result = await client.call_tool(block.name, block.input)
                content, is_error = text_of(result), bool(result.is_error)
            except MCPError as err:                                   # protocol/transport failure, e.g. timeout
                content, is_error = f"MCP error {err.error.code}: {err.error.message}", True
            if verbose:
                flag = "ERROR " if is_error else ""
                print(f"  [{step}] tool  {block.name}({json.dumps(block.input)}) -> {flag}{' '.join(content.split())[:90]}")
            tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": content, "is_error": is_error})
        messages.append({"role": "user", "content": tool_results})
    messages.append({"role": "assistant", "content": [text_block(f"Stopped: reached the {max_iterations}-step limit.")]})
    if verbose:
        print(f"  [guard] stopped after {max_iterations} iterations")
    return messages


def final_text(messages):
    return " ".join(b.text for b in messages[-1]["content"] if getattr(b, "type", None) == "text")


async with Client(weather) as client:
    llm = FakeLLM(umbrella_policy)
    convo = await run_agent(client, llm, [{"role": "user", "content": "Do I need an umbrella in Paris?"}])
    print("\nFINAL:", final_text(convo), f"| model calls: {llm.calls}, messages in history: {len(convo)}")

Read the trace. Round 1 fails (`is_error`), round 2 recovers with `find_city_code`, round 3 chains its output `PAR` into `get_forecast`, and round 4 answers. The tutorial's fixed two-call version would have stopped after the failure.

### 5. Inspect the conversation state
The `messages` list *is* the agent's memory. Here it is as JSON, showing the pairing of every `tool_use` id with a `tool_result`.

In [ ]:
def jsonable(message):
    content = message["content"]
    if isinstance(content, list):
        content = [vars(b) if isinstance(b, SimpleNamespace) else b for b in content]
    return {"role": message["role"], "content": content}


for m in convo[:4]:
    print(json.dumps(jsonable(m))[:260])
print("...")

A follow-up question is appended to the **same** list. That is conversation state across turns. "There" only makes sense because the first question (about Paris) is still in the history. Our scripted policy reads the city from `messages[0]`, while a real model would simply read everything.

In [ ]:
def followup_policy(messages, tool_names):
    question = messages[-1]["content"] if isinstance(messages[-1]["content"], str) else None
    if question and "tomorrow" in question:                     # the model resolves "there" from history
        city = re.search(r"in (\w+)", messages[0]["content"]).group(1)
        return [tool_use("find_city_code", {"city": city})]
    last = tool_trace(messages)[-1]
    if last["name"] == "find_city_code":
        return [tool_use("get_forecast", {"code": last["content"], "day": "tomorrow"})]
    f = json.loads(last["content"])
    return [text_block(f"Tomorrow there: {f['rain_probability']}% rain, {f['celsius']}°C.")]


async with Client(weather) as client:
    convo.append({"role": "user", "content": "And tomorrow there?"})
    convo = await run_agent(client, FakeLLM(followup_policy), convo)
    print("FINAL:", final_text(convo), "| history length:", len(convo))

### 6. Break it: a model that never stops
A buggy model (or a prompt-injected one) can keep calling tools. The iteration guard is what saves your budget.

In [ ]:
def stubborn_policy(messages, tool_names):
    return [tool_use("convert_temperature", {"celsius": len(messages)})]   # always asks for another tool


async with Client(weather) as client:
    llm = FakeLLM(stubborn_policy)
    convo2 = await run_agent(client, llm, [{"role": "user", "content": "Loop forever"}], max_iterations=3)
    print("FINAL:", final_text(convo2), "| model calls:", llm.calls)

### 7. The same loop over a real stdio subprocess
Now the server runs as a separate process, as in the tutorial. We use `Client(stdio_client(StdioServerParameters(...)))`, the tutorial's exact form. The server adds a deliberately slow tool. With `read_timeout_seconds=1`, the slow call raises a timeout `MCPError`, the loop turns it into an `is_error` result, and the model falls back to another tool.

In [ ]:
import sys
import tempfile
import textwrap
from pathlib import Path

from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

workdir = Path(tempfile.mkdtemp())
server_file = workdir / "weather_server.py"
server_file.write_text(textwrap.dedent('''
    import time
    from mcp.server import MCPServer
    from mcp.server.mcpserver.exceptions import ToolError

    server = MCPServer("weather-stdio", log_level="CRITICAL")
    CODES = {"paris": "PAR", "london": "LON"}

    @server.tool()
    def find_city_code(city: str) -> str:
        """Return the 3-letter forecast code for a city name."""
        if city.lower() not in CODES:
            raise ToolError(f"Unknown city {city!r}")
        return CODES[city.lower()]

    @server.tool()
    def live_radar(code: str) -> str:
        """High-resolution live radar (slow)."""
        time.sleep(3)
        return f"radar for {code}: heavy rain"

    @server.tool()
    def get_forecast(code: str) -> dict:
        """Forecast for a city CODE."""
        return {"code": code, "rain_probability": {"PAR": 80, "LON": 95}.get(code, 0)}

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))


def radar_policy(messages, tool_names):
    trace = tool_trace(messages)
    if not trace:
        return [tool_use("find_city_code", {"city": "London"})]
    last = trace[-1]
    if last["name"] == "find_city_code":
        return [tool_use("live_radar", {"code": last["content"]})]
    if last["name"] == "live_radar" and last["is_error"]:
        return [text_block("Radar timed out; using the forecast instead."), tool_use("get_forecast", {"code": "LON"})]
    f = json.loads(last["content"])
    return [text_block(f"Yes: {f['rain_probability']}% chance of rain in London (from {last['name']}).")]


params = StdioServerParameters(command=sys.executable, args=[str(server_file)])
async with Client(stdio_client(params), read_timeout_seconds=1.0) as client:
    print("connected over stdio:", client.server_info.name, "| protocol", client.protocol_version)
    print("tools:", [t.name for t in await list_all_tools(client)])
    convo3 = await run_agent(client, FakeLLM(radar_policy), [{"role": "user", "content": "Is it raining in London?"}])
    print("FINAL:", final_text(convo3))
print("subprocess stopped when the async with block exited")

### 8. Errors that are *not* tool errors
Calling a tool that doesn't exist or passing invalid arguments still returns a result with `isError: true` in this SDK. An over-long call raises instead. A good host handles all three without crashing the chat.

In [ ]:
async with Client(StdioServerParameters(command=sys.executable, args=[str(server_file)])) as client:
    r = await client.call_tool("no_such_tool", {})
    print("unknown tool  -> isError:", r.is_error, "|", text_of(r))
    r = await client.call_tool("find_city_code", {"city": "Atlantis"})
    print("ToolError     -> isError:", r.is_error, "|", text_of(r))
    try:
        await client.call_tool("live_radar", {"code": "PAR"}, read_timeout_seconds=0.5)
    except MCPError as err:
        print("timeout       -> MCPError", err.error.code, err.error.message)

### 9. Optional: the real Claude loop
If `ANTHROPIC_API_KEY` is set, the very same `run_agent` runs with `anthropic.Anthropic()`. Following the tutorial, you'd normally `load_dotenv()` from a git-ignored `.env`. Claude will usually call `find_city_code` first on its own, since the descriptions tell it to.

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

    claude = anthropic.Anthropic()            # reads ANTHROPIC_API_KEY from the environment
    async with Client(weather) as client:
        convo4 = await run_agent(client, claude, [{"role": "user", "content":
            "Do I need an umbrella in London tomorrow? Also give the temperature in Fahrenheit."}])
        print("FINAL:", final_text(convo4))
else:
    print("skipped: set ANTHROPIC_API_KEY to run the same loop with claude-opus-5-5 "
          "(expected: find_city_code -> get_forecast -> convert_temperature -> answer).")

### Your turn
1. Write a policy for *"Is it warmer in Tokyo or London today, in Fahrenheit?"* that requests **two `tool_use` blocks in one turn** (parallel calls) and then `convert_temperature`. Check that your loop sends both results in a single user message.
2. Add a **consent gate** to `run_agent`: before calling any tool whose name starts with `delete_` or `send_`, call an `approve(name, args)` function, and if it returns `False`, send back `is_error: True` with "User denied this action". Test it with a fake destructive tool.
3. Make `run_agent` namespace tools as `weather__get_forecast` when presenting them to the model and strip the prefix before `call_tool`. Why will this matter with two servers?
4. Remove the `try/except MCPError` in `run_agent` and re-run section 7. What happens to the conversation, and why is catching per tool call better than catching per query?